# Baseline 1 — CNN + BiGRU Seizure Detector
**Architecture:** 3-layer 1D CNN → 2-layer Bidirectional GRU → Sigmoid  
**Loss:** Focal Loss (handles 1–5% seizure class imbalance)  
**Evaluation:** Sensitivity, Precision, F1, FP/hour, Temporal Accuracy

In [ ]:
import os, sys, time, json, warnings
warnings.filterwarnings('ignore')

# ── Make src/ importable from notebooks/ ───────────────────────────────────
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, precision_recall_curve)

# ── NeuroScribe src imports ─────────────────────────────────────────────────
from src.data.dataset import EEGDataset, build_train_loader, build_eval_loader
from src.models.cnn_gru import CNNGRUClassifier
from src.models.losses import FocalLoss
from src.training.trainer import run_epoch

sns.set_theme(style='darkgrid'); plt.rcParams['figure.dpi'] = 100

PROCESSED_DIR = '../data/processed'
RESULTS_DIR   = '../results/baseline1'
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED        = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# Hyperparameters
BATCH_SIZE  = 64
LR          = 1e-3
MAX_EPOCHS  = 50
PATIENCE    = 10
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
FS          = 256
WINDOW_SEC  = 4
STEP_SEC    = 2   # 50% overlap → step = 2s

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'Results: {os.path.abspath(RESULTS_DIR)}')

---
## 1. Load Preprocessed Data

In [ ]:
def load_split(split):
    path = os.path.join(PROCESSED_DIR, f'{split}.npz')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Run notebook 01 first to generate: {path}')
    d = np.load(path)
    X, y, pids = d['windows'], d['labels'], d['patient_ids']
    print(f'  {split:<6} → X: {X.shape}  y: {y.shape}  '
          f'seizure: {y.sum():,} ({100*y.mean():.2f}%)  '
          f'patients: {np.unique(pids).tolist()}')
    return X, y, pids

print('Loading splits...')
X_train, y_train, pids_train = load_split('train')
X_val,   y_val,   pids_val   = load_split('val')
X_test,  y_test,  pids_test  = load_split('test')

N_CHANNELS = X_train.shape[1]
print(f'\nChannels: {N_CHANNELS}  |  Window samples: {X_train.shape[2]}')

In [ ]:
# ── Build PyTorch Datasets + DataLoaders via src.data.dataset ──────────────
train_ds = EEGDataset(X_train, y_train)
val_ds   = EEGDataset(X_val,   y_val)
test_ds  = EEGDataset(X_test,  y_test)

train_loader = build_train_loader(train_ds, batch_size=BATCH_SIZE,  num_workers=0, seed=SEED)
val_loader   = build_eval_loader(val_ds,   batch_size=BATCH_SIZE*2, num_workers=0)
test_loader  = build_eval_loader(test_ds,  batch_size=BATCH_SIZE*2, num_workers=0)

print('DataLoaders ready.')
print(f'  Train : {len(train_ds):,} windows  |  {train_ds.n_seizure:,} seizure  ({train_ds.seizure_fraction:.2%})')
print(f'  Val   : {len(val_ds):,} windows  |  {val_ds.n_seizure:,} seizure  ({val_ds.seizure_fraction:.2%})')
print(f'  Test  : {len(test_ds):,} windows  |  {test_ds.n_seizure:,} seizure  ({test_ds.seizure_fraction:.2%})')

---
## 2. Model Definition

In [ ]:
# ── Instantiate model from src.models.cnn_gru ──────────────────────────────
model = CNNGRUClassifier(n_channels=N_CHANNELS).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model      : CNNGRUClassifier  (src.models.cnn_gru)')
print(f'Parameters : {total_params:,}')
print(model)

---
## 3. Focal Loss & Optimiser

In [ ]:
# ── Loss + optimiser (FocalLoss from src.models.losses) ────────────────────
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)
print('Loss      : FocalLoss(alpha=0.75, gamma=2.0)  (src.models.losses)')
print('Optimiser : Adam  lr=1e-3  wd=1e-4')

---
## 4. Training Loop

In [ ]:
# ── Training loop — run_epoch from src.training.trainer ────────────────────
history   = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_val_f1 = 0.0
best_ckpt   = os.path.join(RESULTS_DIR, 'best_model.pt')
no_improve  = 0

print(f'Training for up to {MAX_EPOCHS} epochs (early stop patience={PATIENCE})\n')
print(f'  {"Epoch":>5} | {"Train Loss":>10} | {"Train F1":>9} | {"Val Loss":>9} | {"Val F1":>8} | {"Sens":>7} | {"Prec":>7}')
print('  ' + '-'*72)

for epoch in range(1, MAX_EPOCHS + 1):
    t_res = run_epoch(model, train_loader, criterion, optimizer=optimizer, device=DEVICE)
    v_res = run_epoch(model, val_loader,   criterion, device=DEVICE)

    history['train_loss'].append(t_res['loss'])
    history['val_loss'].append(v_res['loss'])
    history['train_f1'].append(t_res['f1'])
    history['val_f1'].append(v_res['f1'])

    scheduler.step(v_res['f1'])

    marker = ''
    if v_res['f1'] > best_val_f1:
        best_val_f1 = v_res['f1']
        torch.save(model.state_dict(), best_ckpt)
        no_improve = 0
        marker = ' ★'
    else:
        no_improve += 1

    print(f'  {epoch:>5} | {t_res["loss"]:>10.4f} | {t_res["f1"]:>9.4f} | '
          f'{v_res["loss"]:>9.4f} | {v_res["f1"]:>8.4f} | '
          f'{v_res["sensitivity"]:>7.4f} | {v_res["precision"]:>7.4f}{marker}')

    if no_improve >= PATIENCE:
        print(f'\n  Early stop at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs)')
        break

print(f'\nBest val F1: {best_val_f1:.4f}  →  saved to {best_ckpt}')

---
## 5. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs, history['val_loss'],   label='Val',   color='tomato')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Focal Loss')
axes[0].set_title('Loss Curve'); axes[0].legend()

axes[1].plot(epochs, history['train_f1'], label='Train', color='steelblue')
axes[1].plot(epochs, history['val_f1'],   label='Val',   color='tomato')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score Curve'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Threshold Tuning on Validation Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

# Get val probabilities via src.training.trainer.run_epoch
val_res = run_epoch(model, val_loader, criterion, device=DEVICE, threshold=0.5)
val_probs, val_labels = val_res['probs'], val_res['labels']

# Sweep thresholds
thresholds = np.arange(0.1, 0.91, 0.05)
thresh_rows = []
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    tp = ((preds==1)&(val_labels==1)).sum()
    fp = ((preds==1)&(val_labels==0)).sum()
    fn = ((preds==0)&(val_labels==1)).sum()
    sens = tp / max(tp+fn, 1)
    prec = tp / max(tp+fp, 1)
    f1   = 2*sens*prec / max(sens+prec, 1e-8)
    thresh_rows.append({'threshold': round(t,2), 'sensitivity': round(sens,4),
                        'precision': round(prec,4), 'f1': round(f1,4)})

df_thresh = pd.DataFrame(thresh_rows)
best_row  = df_thresh.loc[df_thresh['f1'].idxmax()]
BEST_THRESH = float(best_row['threshold'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_thresh['threshold'], df_thresh['f1'],          label='F1',          color='steelblue', lw=2)
ax.plot(df_thresh['threshold'], df_thresh['sensitivity'], label='Sensitivity',  color='tomato',    lw=1.5, linestyle='--')
ax.plot(df_thresh['threshold'], df_thresh['precision'],   label='Precision',    color='green',     lw=1.5, linestyle='--')
ax.axvline(BEST_THRESH, color='black', linestyle=':', label=f'Best threshold={BEST_THRESH}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Sweep on Validation Set')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'threshold_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Best threshold: {BEST_THRESH}  →  F1={best_row["f1"]:.4f}  '
      f'Sens={best_row["sensitivity"]:.4f}  Prec={best_row["precision"]:.4f}')

---
## 7. Evaluation on Test Set

In [ ]:
test_res    = run_epoch(model, test_loader, criterion, device=DEVICE, threshold=BEST_THRESH)
test_probs  = test_res['probs']
test_labels = test_res['labels']
test_preds  = (test_probs >= BEST_THRESH).astype(int)

tp = ((test_preds==1)&(test_labels==1)).sum()
fp = ((test_preds==1)&(test_labels==0)).sum()
fn = ((test_preds==0)&(test_labels==1)).sum()
tn = ((test_preds==0)&(test_labels==0)).sum()

sensitivity = tp / max(tp+fn, 1)
precision   = tp / max(tp+fp, 1)
specificity = tn / max(tn+fp, 1)
f1          = 2*sensitivity*precision / max(sensitivity+precision, 1e-8)

# FP per hour: each step = STEP_SEC seconds
fp_per_hour = fp * STEP_SEC / 3600

metrics = {
    'Sensitivity (Recall)': round(sensitivity, 4),
    'Precision':            round(precision,   4),
    'Specificity':          round(specificity, 4),
    'F1 Score':             round(f1,          4),
    'FP / hour':            round(fp_per_hour, 2),
    'TP':  int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    'Threshold': BEST_THRESH,
}

print('\n' + '='*45)
print('TEST SET RESULTS — CNN+GRU Baseline')
print('='*45)
for k, v in metrics.items():
    print(f'  {k:<25} {v}')

# Save metrics
with open(os.path.join(RESULTS_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nSaved → {RESULTS_DIR}/metrics.json')

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal','Seizure'], yticklabels=['Normal','Seizure'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix — Test Set')

# ── ROC Curve ─────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc     = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].scatter([metrics['FP']/(metrics['FP']+metrics['TN'])],
                [sensitivity], color='red', zorder=5, label=f'Operating point (t={BEST_THRESH})')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Test Set')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_roc.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Precision-Recall Curve ────────────────────────────────────────────────
prec_curve, rec_curve, _ = precision_recall_curve(test_labels, test_probs)
pr_auc = auc(rec_curve, prec_curve)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rec_curve, prec_curve, color='mediumseagreen', lw=2, label=f'PR AUC = {pr_auc:.4f}')
ax.scatter([sensitivity], [precision], color='red', zorder=5,
           label=f'Operating point (t={BEST_THRESH})')
ax.set_xlabel('Recall (Sensitivity)'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Test Set')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pr_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'PR AUC: {pr_auc:.4f}')

In [ ]:
# ── Per-patient breakdown ─────────────────────────────────────────────────
patient_rows = []
unique_pids  = np.unique(pids_test)

for pid in unique_pids:
    mask = pids_test == pid
    pl, pp = test_labels[mask], test_preds[mask]
    tp_ = ((pp==1)&(pl==1)).sum()
    fp_ = ((pp==1)&(pl==0)).sum()
    fn_ = ((pp==0)&(pl==1)).sum()
    s_  = tp_ / max(tp_+fn_, 1)
    p_  = tp_ / max(tp_+fp_, 1)
    f_  = 2*s_*p_ / max(s_+p_, 1e-8)
    patient_rows.append({
        'Patient':     f'chb{pid:02d}',
        'Windows':     mask.sum(),
        'Sensitivity': round(s_, 4),
        'Precision':   round(p_, 4),
        'F1':          round(f_, 4),
        'FP/hour':     round(fp_ * STEP_SEC / 3600, 2),
    })

df_per_patient = pd.DataFrame(patient_rows)
print('Per-patient results:')
df_per_patient

---
## 8. Final Results Summary

In [ ]:
print('\n' + '='*50)
print('BASELINE 1 — CNN+GRU — FINAL RESULTS')
print('='*50)
print(f'  Model        : 3-layer 1D CNN + 2-layer BiGRU')
print(f'  Loss         : Focal Loss (α={FOCAL_ALPHA}, γ={FOCAL_GAMMA})')
print(f'  Threshold    : {BEST_THRESH}')
print(f'  Split        : patient-independent (18/3/3)')
print()
print(f'  Sensitivity  : {sensitivity:.4f}  ({sensitivity*100:.1f}%)')
print(f'  Precision    : {precision:.4f}  ({precision*100:.1f}%)')
print(f'  F1 Score     : {f1:.4f}')
print(f'  Specificity  : {specificity:.4f}')
print(f'  FP / hour    : {fp_per_hour:.2f}')
print(f'  ROC AUC      : {roc_auc:.4f}')
print(f'  PR  AUC      : {pr_auc:.4f}')
print()
print(f'  Saved to: {os.path.abspath(RESULTS_DIR)}')
print('='*50)